# **Assess Fitness App Usage & Calorie Burn Patterns to Drive Personalized Engagement (FitTech)**

`Context`:
A leading fitness technology platform has curated a comprehensive dataset to analyze user
activity patterns, workout engagement, and app feature usage across multiple regions.
The dataset includes daily workout logs, wearable monitoring data, subscription and
profile records, and in-app engagement sessions. 

`WORKFLOW` :
1. Data Cleaning        ✅
2. Exploratory Data Analysis (EDA)
3. Feature Engineering
4. Merge Datasets
5. KPI Creation
6. Business Insights
7. Dashboard / Presentation

### Project Objectives :

#### Primary Objective : `Assess fitness app usage, workout behavior, calorie burn patterns, and user engagement to generate actionable business insights and support personalized user experiences.`



#### Specific Objectives :

1. Analyze user demographics, subscription plans, fitness goals, and workout preferences.
2. Explore workout activity patterns and identify factors influencing calorie expenditure.
3. Evaluate app engagement behavior, feature adoption, purchase activity, and notification effectiveness.
4. Perform statistical analysis to identify relationships between user characteristics, workout behavior, and engagement metrics.
5. Engineer meaningful features and create business KPIs to measure user performance and engagement.
6. Integrate activity, engagement, and user profile datasets using common user identifiers.
7. Use SQL to perform business-focused querying, aggregation, KPI extraction, and reporting.
8. Use Excel to create pivot tables, summary reports, and stakeholder-friendly analytical views.
9. Build interactive Power BI dashboards to visualize workout trends, engagement patterns, and key business metrics.
10. Develop machine learning models to predict user engagement levels or calorie burn outcomes and support data-driven decision-making.

In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import sqlite3

#The necessary libraries and packages are imported.

# Keep this False while using Run All, so the notebook does not keep creating files.
# Change to True only when you intentionally want to export database, Excel, and model files.
SAVE_OUTPUTS = False

`Data Loading and Inspection`

In [ ]:
activity_df = pd.read_excel("/Volumes/Prasanna/NIIT/The Real Capstone Project/Datasets /Activity.xlsx")
app_engagement_df = pd.read_excel("/Volumes/Prasanna/NIIT/The Real Capstone Project/Datasets /App_Engagement.xlsx")
user_profile_df = pd.read_excel("/Volumes/Prasanna/NIIT/The Real Capstone Project/Datasets /User_Profile.xlsx")

activity_df.info()

In [ ]:
app_engagement_df.info()

In [ ]:
user_profile_df.info()

In [ ]:
activity_df.head()

In [ ]:
app_engagement_df.head()

In [ ]:
user_profile_df.head()

`Dataset Observation` :

- The datasets contain user profile, activity, and app engagement information.
- Numerical and categorical columns are inspected separately before EDA.
- The common key across datasets is `User_ID`.

In [ ]:
activity_num_df = activity_df.select_dtypes(include=['int64', 'float64'])

activity_num_df.head()
activity_num_df.describe() #This command gives us simple stats on the activity table.

From the describe function on the activity_df you can see that `The 75th percentile of calories burned is approximately 524 calories`.

In [ ]:
app_engagement_num_df = app_engagement_df.select_dtypes(include=['int64', 'float64'])

app_engagement_num_df.head()
app_engagement_num_df.describe() #we do the same for the rest two tables as well

### Numerical Data Extraction

## *Data Cleaning*

In [ ]:
activity_df.duplicated().sum()

In [ ]:
app_engagement_df.duplicated().sum()

In [ ]:
user_profile_df.duplicated().sum()

`Out of all three datasets no duplicates,no missing values and no null values were found.`

In [ ]:
activity_df.dtypes

`Data Standardization`

In [ ]:
categorical_cols = [
    'Workout_Type',
    'Workout_Time_of_Day',
    'Device_Used'
]

for col in categorical_cols:
    print(f"\n{'='*50}")
    print(activity_df[col].value_counts(dropna=False))

In [ ]:
activity_df['Activity_ID'].nunique() #checked if all the activity IDs are unique

In [ ]:
activity_df['User_ID'].nunique()

In [ ]:
categorical_cols = [
    'Feature_Used',
    'In_App_Purchase',
    'Notification_Clicked',
    'Workout_Completed'
]

for col in categorical_cols:
    print(f"\n{'='*50}")
    print(app_engagement_df[col].value_counts(dropna=False))

In [ ]:
categorical_cols = [
    'Gender',
    'Age_Group',
    'Region',
    'Subscription_Type',
    'Goal_Type',
    'Preferred_Workout_Type'
]

for col in categorical_cols:  
   print(user_profile_df[col].value_counts(dropna=False)) 

In [ ]:
user_profile_df['Engagement_Level'].head()

`NOTICE` :
`Engagement_Level` is not part of the required EDA schema, so it is removed before analysis.

In [ ]:
user_profile_df = user_profile_df.drop(columns=['Engagement_Level'])

In [ ]:
activity_df.shape
#app_engagement_df.shape
#user_profile_df.shape

`DATA QUALITY SUMMARY`

- Duplicate checks were performed for all three datasets.
- Categorical columns were reviewed for consistency.
- The profile dataset was aligned with the required project schema before EDA.

## *Exploratory Data Analysis*

## User Profile Analysis

### Who uses the app?

In [ ]:
gender_counts = user_profile_df['Gender'].value_counts()

plt.figure(figsize=(6, 6))
plt.pie(
    gender_counts,
    labels=gender_counts.index, # type: ignore
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'width': 0.4}
)
plt.title('User Distribution by Gender')
plt.show()

**Insight:** Male users form the largest gender segment with 277 users, about 46.2% of the user base.

### Which age groups use the platform most frequently?

In [ ]:
age_order = ['18-25', '26-35', '36-45', '46+']

plt.figure(figsize=(8, 5))
sns.countplot(
    data=user_profile_df,
    x='Age_Group',
    order=age_order
)
plt.title('User Count by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Number of Users')
plt.show()

**Insight:** The 26-35 age group has the highest usage with 244 users, representing about 40.7% of users.

### Which regions contribute the most users?

In [ ]:
region_counts = user_profile_df['Region'].value_counts()

plt.figure(figsize=(8, 5))
sns.barplot(
    x=region_counts.values,
    y=region_counts.index
)
plt.title('Users by Region')
plt.xlabel('Number of Users')
plt.ylabel('Region')
plt.show()

**Insight:** Mumbai contributes the most users with 108 users, accounting for 18.0% of the profile dataset.

### Which subscription plans are most popular?

In [ ]:
subscription_counts = user_profile_df['Subscription_Type'].value_counts()

plt.figure(figsize=(6, 6))
plt.pie(
    subscription_counts,
    labels=subscription_counts.index, # type: ignore
    autopct='%1.1f%%',
    startangle=90
)
plt.title('Subscription Plan Share')
plt.show()

**Insight:** The Free plan is the most popular subscription type with 293 users, about 48.8% of users.

### What are users trying to achieve?

In [ ]:
goal_percent = user_profile_df['Goal_Type'].value_counts(normalize=True) * 100
goal_percent_df = pd.DataFrame([goal_percent.values], columns=goal_percent.index)

plt.figure(figsize=(9, 2.5))
sns.heatmap(
    goal_percent_df,
    annot=True,
    fmt='.1f',
    cmap='YlGnBu'
)
plt.title('User Goal Distribution (%)')
plt.xlabel('Goal Type')
plt.yticks([])
plt.show()

**Insight:** Fitness is the leading user goal with 163 users, representing about 27.2% of the user base.

### Which workouts are preferred by users?

In [ ]:
preferred_workout_percent = user_profile_df['Preferred_Workout_Type'].value_counts(normalize=True) * 100

plt.figure(figsize=(8, 5))
sns.barplot(
    x=preferred_workout_percent.index,
    y=preferred_workout_percent.values
)
plt.title('Preferred Workout Type Share')
plt.xlabel('Preferred Workout Type')
plt.ylabel('Percentage of Users')
plt.xticks(rotation=30)
plt.show()

**Insight:** Yoga is the most preferred workout type with 159 users, about 26.5% of users.

## Activity Analysis

### Which workout types are most popular?

In [ ]:
workout_order = activity_df['Workout_Type'].value_counts().index

plt.figure(figsize=(8, 5))
sns.countplot(
    data=activity_df,
    x='Workout_Type',
    order=workout_order
)
plt.title('Workout Activity Count by Type')
plt.xlabel('Workout Type')
plt.ylabel('Number of Activities')
plt.xticks(rotation=30)
plt.show()

**Insight:** Yoga is the most recorded workout type with 25,115 activity records, about 25.1% of all workouts.

### Which workout types burn the most calories?

In [ ]:
avg_calories = activity_df.groupby('Workout_Type')['Calories_Burned'].mean().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(
    x=avg_calories.values,
    y=avg_calories.index
)
plt.title('Average Calories Burned by Workout Type')
plt.xlabel('Average Calories Burned')
plt.ylabel('Workout Type')
plt.show()

**Insight:** Cardio has the highest average calorie burn at about 539.7 calories per workout.

### When do users exercise?

In [ ]:
time_counts = activity_df['Workout_Time_of_Day'].value_counts()

plt.figure(figsize=(7, 4))
sns.barplot(
    x=time_counts.values,
    y=time_counts.index
)
plt.title('Workout Count by Time of Day')
plt.xlabel('Number of Workouts')
plt.ylabel('Workout Time of Day')
plt.show()

**Insight:** Morning is the most common workout time with 45,089 sessions, about 45.1% of all workouts.

### Which devices are most commonly used?

In [ ]:
device_counts = activity_df['Device_Used'].value_counts()

plt.figure(figsize=(6, 6))
plt.pie(
    device_counts,
    labels=device_counts.index,
    autopct='%1.1f%%',
    startangle=90
)
plt.title('Device Usage Share')
plt.show()

**Insight:** Mobile is the most commonly used device with 33,455 workout records, about 33.5% of activity entries.

### How are calories burned distributed?

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(
    data=activity_df,
    x='Calories_Burned',
    bins=30
)
plt.title('Distribution of Calories Burned')
plt.xlabel('Calories Burned')
plt.ylabel('Number of Activities')
plt.show()

**Insight:** Calories burned has a median of 357 calories, with the middle 50% of workouts between 236 and 524 calories.

### How is workout duration distributed?

In [ ]:
plt.figure(figsize=(8, 5))
sns.kdeplot(
    data=activity_df,
    x='Duration_Minutes',
    fill=True
)
plt.title('Workout Duration Density')
plt.xlabel('Duration in Minutes')
plt.ylabel('Density')
plt.show()

**Insight:** Workout duration centers around 60 minutes, with the middle 50% of sessions between 40 and 80 minutes.

### How are daily steps distributed?

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(
    data=activity_df,
    x='Steps_Count'
)
plt.title('Distribution of Steps Count')
plt.xlabel('Steps Count')
plt.show()

**Insight:** Steps count has a median of 0 and a wide upper spread, with the top quartile starting around 5,513 steps.

## Engagement Analysis

### Which features are used most?

In [ ]:
feature_counts = app_engagement_df['Feature_Used'].value_counts()

plt.figure(figsize=(8, 5))
sns.barplot(
    x=feature_counts.values,
    y=feature_counts.index
)
plt.title('Most Used App Features')
plt.xlabel('Number of Sessions')
plt.ylabel('Feature Used')
plt.show()

**Insight:** Workout Tracker is the most used feature with 79,565 sessions, about 39.8% of app engagement records.

### Are notifications effective?

In [ ]:
notification_counts = app_engagement_df['Notification_Clicked'].value_counts()

plt.figure(figsize=(6, 6))
plt.pie(
    notification_counts,
    labels=notification_counts.index, # type: ignore
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'width': 0.4}
)
plt.title('Notification Click Share')
plt.show()

**Insight:** Notifications were clicked in 73,024 sessions, giving a click rate of about 36.5%.

### Are users completing workouts?

In [ ]:
completion_counts = app_engagement_df['Workout_Completed'].value_counts()

plt.figure(figsize=(6, 6))
plt.pie(
    completion_counts,
    labels=completion_counts.index, # type: ignore
    autopct='%1.1f%%',
    startangle=90
)
plt.title('Workout Completion Share')
plt.show()

**Insight:** Users completed workouts in 150,089 sessions, giving a completion rate of about 75.0%.

### Are users making in-app purchases?

In [ ]:
purchase_order = app_engagement_df['In_App_Purchase'].value_counts().index

plt.figure(figsize=(6, 5))
sns.countplot(
    data=app_engagement_df,
    x='In_App_Purchase',
    order=purchase_order
)
plt.title('In-App Purchase Count')
plt.xlabel('In-App Purchase')
plt.ylabel('Number of Sessions')
plt.show()

**Insight:** In-app purchases occurred in 27,661 sessions, about 13.8% of engagement records.

### How is session duration distributed?

In [ ]:
plt.figure(figsize=(8, 4))
sns.violinplot(
    data=app_engagement_df,
    x='Session_Duration_Minutes'
)
plt.title('Session Duration Distribution')
plt.xlabel('Session Duration in Minutes')
plt.show()

**Insight:** Session duration has a median of 18 minutes, with the middle 50% of sessions between 14 and 22 minutes.

## Objective 4: Basic Statistical Analysis

**Question:** Which numerical workout variables are related to each other?

In [ ]:
activity_corr = activity_df[['Duration_Minutes', 'Calories_Burned', 'Steps_Count','Heart_Rate_Avg']].corr()

plt.figure(figsize=(6, 4))
sns.heatmap(activity_corr, annot=True, cmap='Blues')
plt.title('Correlation Between Activity Metrics')
plt.show()

**Insight:** This helps check whether longer workouts, higher calorie burn and step count move together.

**Question:** Which workout type has the highest average calorie burn?

In [ ]:
avg_calories_by_workout = activity_df.groupby('Workout_Type')['Calories_Burned'].mean().sort_values(ascending=False)

avg_calories_by_workout

**Insight:** This simple groupby shows which workout type burns more calories on average.

**Question:** Which app features have the longest average session duration?

In [ ]:
avg_session_by_feature = app_engagement_df.groupby('Feature_Used')['Session_Duration_Minutes'].mean().sort_values(ascending=False)

avg_session_by_feature

**Insight:** This shows which features keep users engaged for more time.

**Question:** How are subscription type and fitness goal related?

In [ ]:
subscription_goal_table = pd.crosstab(
    user_profile_df['Subscription_Type'],
    user_profile_df['Goal_Type']
)

subscription_goal_table

**Insight:** This table helps compare user goals across different subscription plans.

### Hypothesis Testing

In [ ]:
from scipy import stats

**Test 1: Do different workout types burn different average calories?**

- Null Hypothesis (H0): Average calories burned is the same for all workout types.
- Alternate Hypothesis (H1): At least one workout type has a different average calorie burn.

In [ ]:
cardio = activity_df[activity_df['Workout_Type'] == 'Cardio']['Calories_Burned']
strength = activity_df[activity_df['Workout_Type'] == 'Strength']['Calories_Burned']
yoga = activity_df[activity_df['Workout_Type'] == 'Yoga']['Calories_Burned']
mix = activity_df[activity_df['Workout_Type'] == 'Mix']['Calories_Burned']

f_stat, p_value = stats.f_oneway(cardio, strength, yoga, mix)

print('F-statistic:', round(f_stat, 2))
print('P-value:', p_value)

if p_value < 0.05:
    print('Conclusion: Reject H0. Average calories burned differs by workout type.')
else:
    print('Conclusion: Fail to reject H0. No significant difference found.')

**Insight:** This test checks whether workout type has a statistically significant relationship with calorie burn.

**Test 2: Are notification clicks and workout completion related?**

- Null Hypothesis (H0): Notification clicks and workout completion are independent.
- Alternate Hypothesis (H1): Notification clicks and workout completion are related.

In [ ]:
notification_completion_table = pd.crosstab(
    app_engagement_df['Notification_Clicked'],
    app_engagement_df['Workout_Completed']
)

chi2, p_value, dof, expected = stats.chi2_contingency(notification_completion_table)

print(notification_completion_table)
print('Chi-square statistic:', round(chi2, 2)) # type: ignore
print('P-value:', p_value)

if p_value < 0.05: # type: ignore
    print('Conclusion: Reject H0. Notification clicks and workout completion are related.')
else:
    print('Conclusion: Fail to reject H0. No significant relationship found.')

**Insight:** This test helps understand whether notification engagement is connected with workout completion.

**Test 3: Do clicked notifications have a different average session duration?**

- Null Hypothesis (H0): Average session duration is the same for clicked and not-clicked notifications.
- Alternate Hypothesis (H1): Average session duration is different for clicked and not-clicked notifications.

In [ ]:
clicked_sessions = app_engagement_df[app_engagement_df['Notification_Clicked'] == 'Yes']['Session_Duration_Minutes']
not_clicked_sessions = app_engagement_df[app_engagement_df['Notification_Clicked'] == 'No']['Session_Duration_Minutes']

t_stat, p_value = stats.ttest_ind(clicked_sessions, not_clicked_sessions, equal_var=False)

print('T-statistic:', round(t_stat, 2)) # type: ignore
print('P-value:', p_value)

if p_value < 0.05: # type: ignore
    print('Conclusion: Reject H0. Session duration differs based on notification click status.')
else:
    print('Conclusion: Fail to reject H0. No significant difference found.')

**Insight:** This test checks whether users who click notifications spend a different amount of time in the app.

### Validation Summary

In [ ]:
anova_p_value = stats.f_oneway(cardio, strength, yoga, mix).pvalue

chi2_table = pd.crosstab(
    app_engagement_df['Notification_Clicked'],
    app_engagement_df['Workout_Completed']
)
chi2_p_value = stats.chi2_contingency(chi2_table)[1]

ttest_p_value = stats.ttest_ind(
    clicked_sessions,
    not_clicked_sessions,
    equal_var=False
).pvalue

validation_summary = pd.DataFrame({
    'Test': [
        'ANOVA - Calories by Workout Type',
        'Chi-square - Notification Click vs Completion',
        'T-test - Session Duration by Notification Click'
    ],
    'P_Value': [anova_p_value, chi2_p_value, ttest_p_value],
    'Decision': [
        'Significant' if anova_p_value < 0.05 else 'Not Significant',
        'Significant' if chi2_p_value < 0.05 else 'Not Significant',
        'Significant' if ttest_p_value < 0.05 else 'Not Significant'
    ]
})

validation_summary

In [ ]:
if SAVE_OUTPUTS:
    validation_summary.to_excel(
        'Validation_Summary.xlsx',
        index=False
    )
    print('Validation_Summary.xlsx saved.')
else:
    print('File export skipped. Set SAVE_OUTPUTS = True to save Validation_Summary.xlsx.')

**Insight:** This creates a simple validation report showing which statistical tests are significant.

## Objective 5: Basic Feature Engineering

**Note:** Simple feature engineering can be done before merging. Features that need columns from multiple datasets should be created after objective 6, when the datasets are merged.

**Activity Features**

In [ ]:
activity_df['Calories_Per_Minute'] = activity_df['Calories_Burned'] / activity_df['Duration_Minutes']

activity_df['Duration_Category'] = pd.cut(
    activity_df['Duration_Minutes'],
    bins=[0, 30, 60, 100],
    labels=['Short', 'Medium', 'Long']
)

activity_df['Steps_Category'] = pd.cut(
    activity_df['Steps_Count'],
    bins=[-1, 3000, 7000, 12000],
    labels=['Low Steps', 'Medium Steps', 'High Steps']
)

activity_df[['Duration_Minutes', 'Calories_Burned', 'Calories_Per_Minute', 'Duration_Category', 'Steps_Count', 'Steps_Category']].head()

**Insight:** These columns make workout intensity and activity level easier to compare.

**Engagement Features**

In [ ]:
app_engagement_df['Notification_Clicked_Flag'] = app_engagement_df['Notification_Clicked'].map({'Yes': 1, 'No': 0})
app_engagement_df['Workout_Completed_Flag'] = app_engagement_df['Workout_Completed'].map({'Yes': 1, 'No': 0})
app_engagement_df['Purchase_Flag'] = app_engagement_df['In_App_Purchase'].map({'Yes': 1, 'No': 0})

app_engagement_df['Session_Duration_Category'] = pd.cut(
    app_engagement_df['Session_Duration_Minutes'],
    bins=[0, 10, 20, 30],
    labels=['Short', 'Medium', 'Long']
)

app_engagement_df[['Session_Duration_Minutes', 'Session_Duration_Category', 'Notification_Clicked_Flag', 'Workout_Completed_Flag', 'Purchase_Flag']].head()

**Insight:** These flag columns make yes/no engagement behavior easier to calculate later.

**User Profile Features**

In [ ]:
user_profile_df['Age_Group_Number'] = user_profile_df['Age_Group'].map({
    '18-25': 1,
    '26-35': 2,
    '36-45': 3,
    '46+': 4
})

user_profile_df['Paid_User_Flag'] = (
    user_profile_df['Subscription_Type']
    .map({
        'Free': 0,
        'Trial': 0,
        'Premium': 1
    })
)

user_profile_df[['Age_Group', 'Age_Group_Number', 'Subscription_Type', 'Paid_User_Flag']].head()

**Insight:** These simple profile features can help later analysis, dashboards, and basic modeling.

## Objective 6: Merging the Datasets

In [ ]:
activity_summary = (
    activity_df.groupby('User_ID').agg(
        Total_Workouts=('Activity_ID','count'),
        Total_Calories=('Calories_Burned','sum'),
        Avg_Calories=('Calories_Burned','mean'),
        Avg_Duration=('Duration_Minutes','mean'),
        Total_Steps=('Steps_Count','sum')
    )
    .reset_index()
)
activity_summary

In [ ]:
app_engagement_summary = (
    app_engagement_df.groupby('User_ID').agg(
        Total_Sessions=('Session_ID','count'),
        Avg_Session_Duration=('Session_Duration_Minutes','mean'),
        Total_Session_Time=('Session_Duration_Minutes','sum'),
        Completion_Rate=('Workout_Completed_Flag','mean'),
        Notification_Click_Rate=('Notification_Clicked_Flag','mean'),
        Purchase_Rate=('Purchase_Flag','mean')
    )
)
app_engagement_summary

In [ ]:
final_df = (
    user_profile_df
    .merge(activity_summary,on='User_ID')
    .merge(app_engagement_summary,on='User_ID')
)
pd.set_option('display.max_columns', None)

final_df.head()

## Objective 7: SQLite Analysis

In [ ]:
import sqlite3

if SAVE_OUTPUTS:
    conn = sqlite3.connect('fittech.db')
    print('Connected to fittech.db file.')
else:
    conn = sqlite3.connect(':memory:')
    print('Connected to temporary in-memory SQLite database.')

cursor = conn.cursor()

In [ ]:
user_profile_df.to_sql(
    'user_profile',
    conn,
    if_exists='replace',
    index=False
)
activity_df.to_sql(
    'activity',
    conn,
    if_exists='replace',
    index=False
)
app_engagement_df.to_sql(
    'app_engagement',
    conn,
    if_exists='replace',
    index=False
)

activity_summary.to_sql(
    'activity_summary',
    conn,
    if_exists='replace',
    index=False
)
app_engagement_summary.to_sql(
    'engagement_summary',
    conn,
    if_exists='replace',
    index=False
)
final_df.to_sql(
    'final_user_summary',
    conn,
    if_exists='replace',
    index=False
)

In [ ]:
query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

pd.read_sql(query, conn)


Query 1: Top 10 Most Active Users

In [ ]:
query = """
SELECT
    User_ID,
    Total_workouts,
    Total_Calories
FROM final_user_summary
ORDER BY Total_workouts DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

Query 2: Average Calories Burned by Subscription Type

In [ ]:
query = """
SELECT
    Subscription_Type,
    ROUND(AVG(Total_Calories),2) AS Avg_Calories
FROM final_user_summary
GROUP BY Subscription_Type
ORDER BY Avg_Calories DESC;
"""

pd.read_sql(query, conn)

Query 3: Average Session Duration by Region

In [ ]:
query = """
SELECT
    Region,
    ROUND(AVG(Avg_Session_Duration), 2) AS Avg_Session_Duration
FROM final_user_summary
GROUP BY Region
ORDER BY Avg_Session_Duration DESC;
"""

pd.read_sql(query, conn)

Query 4: Purchase Rate by Subscription Plan

In [ ]:
query = """
SELECT
    Subscription_Type,
    ROUND(AVG(Purchase_Rate) * 100, 2) AS Purchase_Percentage
FROM final_user_summary
GROUP BY Subscription_Type
ORDER BY Purchase_Percentage DESC;
"""

pd.read_sql(query, conn)

Query 5: Workout Completion Rate by Goal Type

In [ ]:
query = """
SELECT
    Goal_Type,
    ROUND(AVG(Completion_Rate)*100,2) AS Completion_Percentage
FROM final_user_summary
GROUP BY Goal_Type
ORDER BY Completion_Percentage DESC;
"""

pd.read_sql(query, conn)

Query 6: High Engagement Users

In [ ]:
query = """
SELECT
    User_ID,
    Total_Sessions,
    Total_Session_Time
FROM final_user_summary
ORDER BY Total_Sessions DESC
LIMIT 20;
"""

pd.read_sql(query, conn)

In [ ]:
conn.close()

In [ ]:
if SAVE_OUTPUTS:
    final_df.to_excel(
        'FitTech_Final_Dataset.xlsx',
        index=False
    )
    print('FitTech_Final_Dataset.xlsx saved.')
else:
    print('File export skipped. Set SAVE_OUTPUTS = True to save FitTech_Final_Dataset.xlsx.')

## Objective 10: Machine Learning

**Goal:** Use simple machine learning models to support the business story.

In this section, we build two types of models for a clear business purpose:

- **Classification:** Predict whether a user is highly engaged or not. This helps the business identify users who are likely to stay active and target retention or reward campaigns.
- **Regression:** Predict each user's total session time. This helps estimate value and engagement level quantitatively, so the company can focus on users who spend more time in the app.

A good evaluation score is important because it shows the model can generalize beyond the examples it was trained on. We will use separate training and test sets and report accuracy for classification, and MAE/RMSE/R² for regression.


**Step 1: Create the classification target**

In [ ]:
ml_df = final_df.copy()

median_session_time = ml_df['Total_Session_Time'].median()

ml_df['High_Engagement_Flag'] = np.where(
    ml_df['Total_Session_Time'] >= median_session_time,
    1,
    0
)

ml_df[['User_ID', 'Total_Session_Time', 'High_Engagement_Flag']].head()

**Insight:** Users with session time greater than or equal to the median are marked as high engagement users.

**Step 2: Select input columns**

In [ ]:
ml_features = [
    'Gender',
    'Age_Group',
    'Region',
    'Subscription_Type',
    'Goal_Type',
    'Preferred_Workout_Type',
    'Total_Workouts',
    'Avg_Calories',
    'Avg_Duration',
    'Total_Steps',
    'Completion_Rate',
    'Notification_Click_Rate',
    'Purchase_Rate'
]

X = ml_df[ml_features]
y_classification = ml_df['High_Engagement_Flag']
y_regression = ml_df['Total_Session_Time']

X.head()

**Insight:** The same input columns are used for both stories: engagement classification and session-time prediction.

**Step 3: Encode categorical columns**

In [ ]:
from sklearn.preprocessing import OneHotEncoder

X_encoded = X.copy()

age_group_order = {
    '18-25': 1,
    '26-35': 2,
    '36-45': 3,
    '46+': 4
}

X_encoded['Age_Group'] = X_encoded['Age_Group'].map(age_group_order)

categorical_columns = [
    'Gender',
    'Region',
    'Subscription_Type',
    'Goal_Type',
    'Preferred_Workout_Type'
]

encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

encoded_array = encoder.fit_transform(X_encoded[categorical_columns])
encoded_columns = encoder.get_feature_names_out(categorical_columns)

encoded_df = pd.DataFrame(
    encoded_array,
    columns=encoded_columns,
    index=X_encoded.index
)

X = pd.concat(
    [X_encoded.drop(columns=categorical_columns), encoded_df],
    axis=1
)

X.head()

**Insight:** `Age_Group` is label encoded because it has a natural order. Other text columns are one-hot encoded because they do not have a meaningful ranking.

**Step 4: Split the data into training and testing data**

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_class_train, y_class_test, y_reg_train, y_reg_test = train_test_split(
    X,
    y_classification,
    y_regression,
    test_size=0.2,
    random_state=42,
    stratify=y_classification
)

print('Training rows:', X_train.shape[0])
print('Testing rows:', X_test.shape[0])

**Insight:** The model learns from training data and is checked on separate testing data.

**Step 5: Apply StandardScaler**

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

**Insight:** StandardScaler puts numerical values on a similar scale. This is important for models like Logistic Regression, KNN, and Linear Regression.

**Why evaluation matters:**

We split the data into training and test sets so that the model is evaluated on data it has never seen before. This helps us understand whether the model is truly useful for making predictions in the real world, not just memorizing the training data.


**Step 6: Build classification models**

**Step 6.1: Build Logistic Regression model**


In [ ]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(max_iter=1000)
logistic_model.fit(X_train_scaled, y_class_train)
logistic_predictions = logistic_model.predict(X_test_scaled)


**Step 6.2: Build K-Nearest Neighbors model**


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_class_train)
knn_predictions = knn_model.predict(X_test_scaled)


**Step 6.3: Build Decision Tree model**


In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_model.fit(X_train_scaled, y_class_train)
tree_predictions = tree_model.predict(X_test_scaled)


**Step 7: Compare classification models**


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

classification_results = pd.DataFrame({
    'Model': ['Logistic Regression', 'KNN', 'Decision Tree'],
    'Accuracy': [
        accuracy_score(y_class_test, logistic_predictions),
        accuracy_score(y_class_test, knn_predictions),
        accuracy_score(y_class_test, tree_predictions)
    ]
})

classification_results


**Insight:** Accuracy helps compare which classification model performs better on the test data. A good evaluation score means the chosen model predicts high engagement reliably on unseen users and manages both classes well.


**Step 8: Check the best classification model in detail**


In [ ]:
best_classification_model_name = classification_results.sort_values(
    by='Accuracy',
    ascending=False
).iloc[0]['Model']

if best_classification_model_name == 'Logistic Regression':
    best_classification_model = logistic_model
    best_classification_predictions = logistic_predictions
elif best_classification_model_name == 'KNN':
    best_classification_model = knn_model
    best_classification_predictions = knn_predictions
else:
    best_classification_model = tree_model
    best_classification_predictions = tree_predictions

print('Best Classification Model:', best_classification_model_name)
print()
print('Confusion Matrix:')
print(confusion_matrix(y_class_test, best_classification_predictions))
print()
print('Classification Report:')
print(classification_report(y_class_test, best_classification_predictions))


**Insight:** The confusion matrix and classification report show how well the best model predicts both engagement groups.


In [ ]:
best_confusion_matrix = confusion_matrix(y_class_test, best_classification_predictions)

plt.figure(figsize=(5, 4))
sns.heatmap(
    best_confusion_matrix,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Not High Engagement', 'High Engagement'],
    yticklabels=['Not High Engagement', 'High Engagement']
)
plt.title('Confusion Matrix - ' + best_classification_model_name)
plt.xlabel('Predicted Value')
plt.ylabel('Actual Value')
plt.show()

**Step 9: Build a Linear Regression model**

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

linear_model = LinearRegression()
linear_model.fit(X_train_scaled, y_reg_train)
linear_predictions = linear_model.predict(X_test_scaled)


**Step 10: Evaluate the Linear Regression model**

A good regression score has a low MAE and RMSE with a high R². This means the model prediction of total session time is close to the actual values and captures the underlying engagement pattern.


In [ ]:
mae = mean_absolute_error(y_reg_test, linear_predictions)
rmse = np.sqrt(mean_squared_error(y_reg_test, linear_predictions))
r2 = r2_score(y_reg_test, linear_predictions)

regression_results = pd.DataFrame({
    'Model': ['Linear Regression'],
    'MAE': [mae],
    'RMSE': [rmse],
    'R2_Score': [r2]
})

regression_results


**Step 11: Check important features**

In [ ]:
if best_classification_model_name == 'Logistic Regression':
    feature_importance = pd.DataFrame({
        'Feature': X.columns,
        'Coefficient': logistic_model.coef_[0]
    })

    feature_importance['Importance'] = feature_importance['Coefficient'].abs()
    feature_importance = feature_importance.sort_values(by='Importance', ascending=False)

elif best_classification_model_name == 'Decision Tree':
    feature_importance = pd.DataFrame({
        'Feature': X.columns,
        'Importance': tree_model.feature_importances_
    }).sort_values(by='Importance', ascending=False)

else:
    feature_importance = pd.DataFrame({
        'Feature': X.columns,
        'Importance': 0
    })

feature_importance.head(10)

**Insight:** Feature importance helps explain which columns are most useful for the model story.

**Step 12: Visualisation of the important Features**

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(
data=feature_importance.head(10),
x="Importance",
y="Feature"
)

plt.title("Top 10 Important Features")

**Step 12: Save the model outputs**

In [ ]:
if SAVE_OUTPUTS:
    import joblib

    joblib.dump(best_classification_model, 'Best_Engagement_Classification_Model.pkl')
    joblib.dump(linear_model, 'Linear_Regression_Session_Time_Model.pkl')
    joblib.dump(scaler, 'Standard_Scaler.pkl')
    joblib.dump(encoder, 'One_Hot_Encoder.pkl')

    with pd.ExcelWriter('ML_Model_Results.xlsx') as writer:
        classification_results.to_excel(writer, sheet_name='Classification Results', index=False)
        regression_results.to_excel(writer, sheet_name='Regression Results', index=False)
        feature_importance.head(10).to_excel(writer, sheet_name='Top Features', index=False)

    print('Saved classification model, linear regression model, scaler, encoder, and ML results file.')
else:
    print('Model/file export skipped. Set SAVE_OUTPUTS = True to save ML output files.')

**Insight:** Saving the models, scaler, and encoder makes the work reusable and easier to explain in the final presentation.

**Final ML Story:**

The classification models help identify users who are likely to be highly engaged. This can support retention campaigns and personalized engagement strategies.

The Linear Regression model predicts total session time. This gives a numerical view of engagement and helps understand what may increase or reduce user time spent in the app.


## Business Summary and Recommendations

### Business Summary

This project helped me understand how users are using the fitness app, how they workout, and how they engage with app features.

The main areas covered in the analysis were:

- User profile analysis using gender, age group, region, subscription type, goal type, and preferred workout type.
- Activity analysis using workout type, calories burned, duration, steps, time of day, and device used.
- Engagement analysis using feature usage, notification clicks, workout completion, purchases, and session duration.
- SQL analysis for creating business-focused summary views.
- Basic hypothesis testing to validate a few assumptions using statistics.
- Machine learning models to predict high engagement users and total session time.

### Key Findings

- Yoga appears often in workout activity and user preference, so it is an important workout category for the app.
- Cardio burns the highest average calories, so it can be promoted for users focused on calorie burn.
- Morning is the most common workout time, which can help decide when to send reminders.
- Workout Tracker is the most used app feature, showing that users value progress tracking.
- Workout completion is fairly strong, but notification clicks and purchases can still be improved.
- Subscription-level reports show differences between user groups, which can help with marketing and retention decisions.

### Recommendations

- Send reminders around morning hours because many users workout during that time.
- Promote Cardio workouts for users whose goal is calorie burn or fitness improvement.
- Improve notification messages because not every user is clicking notifications.
- Focus on users with low completion rates and low notification click rates for re-engagement campaigns.
- Use subscription reports to compare Free, Trial, and Premium users before planning upgrade campaigns.
- Keep improving the Workout Tracker feature because it is one of the most used features.

### Conclusion

The analysis shows that fitness app data can be used to understand user behaviour and support better business decisions.

EDA helped identify usage patterns. SQL helped create business-ready reports. Hypothesis testing helped validate relationships in the data. Feature engineering and machine learning gave a basic way to identify highly engaged users and estimate session time.

Overall, the project gives a clear starting point for improving user engagement, workout completion, and subscription strategy.

### Limitations and Challenges

- The analysis is based only on the available dataset columns.
- The machine learning models are basic and should be treated as a first attempt, not final production models.
- Some recommendations should be tested with real users before applying them fully.
- More time-based analysis can be done later using the date columns.
- More advanced models may improve prediction results, but I kept the approach simple for this project stage.

### Future Scope

- Build a complete Power BI dashboard using the exported dashboard tables.
- Add monthly and weekly trend analysis using date columns.
- Try more machine learning models and compare their performance.
- Create personalized recommendation rules for different user segments.
- Add retention or churn analysis if future user activity data is available.